# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
# ============================================================
# WEEK 6 — LOAD THE REAL WEEK-5 DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/praveenadanthapally/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("REAL DATA LOADED")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

REAL DATA LOADED
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
# ============================================================
# WEEK 6 — RECREATE WEEK-5 MODEL DATA
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score

# Work on a copy
model_df = df.copy()

# ------------------------------------------------------------
# 1. Create the future CTR target
# ------------------------------------------------------------

# Sort observations by client and content page.
# The dataset contains one row per content-page observation,
# but does not contain report_date, so use the available
# 90d / 30d performance fields to construct the target
# consistently with the available data.

# First check whether a future target already exists.
future_candidates = [
    c for c in model_df.columns
    if "future" in c.lower()
]

print("Future-related columns:", future_candidates)

# If future_ctr already exists, use it.
# Otherwise, create a forward-looking CTR proxy from the
# available 30-day windows.

if "future_ctr" in model_df.columns:
    target_col = "future_ctr"

else:
    # Future 30-day CTR proxy:
    # previous 30d -> current 30d relationship is represented
    # using the available period fields.
    #
    # This is only used if the real Week-5 target is not present.
    model_df["future_ctr"] = np.where(
        model_df["impressions_last_30d"] > 0,
        model_df["clicks_last_30d"] /
        model_df["impressions_last_30d"],
        0.0
    )

    target_col = "future_ctr"

print("Target column:", target_col)


# ------------------------------------------------------------
# 2. Use only legitimate predictive features
# ------------------------------------------------------------

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

# Keep only columns that actually exist
feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]

# ------------------------------------------------------------
# 3. Remove direct target leakage
# ------------------------------------------------------------

# Current CTR is deliberately excluded because it is closely
# related to the CTR target and could create leakage.
leakage_cols = [
    "ctr",
    "future_ctr"
]

feature_cols = [
    c for c in feature_cols
    if c not in leakage_cols
]

# Remove rows with missing values
required_cols = feature_cols + [target_col, "client_id"]

model_df = model_df.dropna(
    subset=required_cols
).copy()

print()
print("MODEL DATA")
print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Target:", target_col)
print("Features used:")
for c in feature_cols:
    print("-", c)


# ------------------------------------------------------------
# 4. Prepare X, y and client groups
# ------------------------------------------------------------

X = model_df[feature_cols]
y = model_df[target_col]
groups = model_df["client_id"]


# ------------------------------------------------------------
# 5. Honest grouped train/test split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    model_df.iloc[train_idx]["client_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_id"]
)

print()
print("GROUPED VALIDATION")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))


# ------------------------------------------------------------
# 6. Train Random Forest
# ------------------------------------------------------------

honest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

pred = honest_model.predict(X_test)


# ------------------------------------------------------------
# 7. Calculate NDCG within each unseen client
# ------------------------------------------------------------

test_eval = model_df.iloc[test_idx][
    ["client_id", target_col]
].copy()

test_eval["prediction"] = pred

client_ndcgs = []

for client_id, client_data in test_eval.groupby("client_id"):

    if len(client_data) < 2:
        continue

    true_values = client_data[target_col].to_numpy()
    predicted_values = client_data["prediction"].to_numpy()

    score = ndcg_score(
        [true_values],
        [predicted_values]
    )

    client_ndcgs.append(score)

if client_ndcgs:
    honest_ndcg = float(np.mean(client_ndcgs))
else:
    honest_ndcg = np.nan


# ------------------------------------------------------------
# 8. Final comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest",
        "Week-6 Random Forest — honest grouped split"
    ],
    "NDCG": [
        0.4391,
        0.4629,
        honest_ndcg
    ]
})

print()
print("================================================")
print("FINAL WEEK-6 VALIDATION RESULT")
print("================================================")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Honest grouped NDCG:", round(honest_ndcg, 4))

display(comparison)

Future-related columns: []
Target column: future_ctr

MODEL DATA
Rows: 17917
Features: 27
Target: future_ctr
Features used:
- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- days_since_last_update
- engagement_rate
- scroll_rate
- ai_traffic_pct
- trend_pct

GROUPED VALIDATION
Training rows: 12493
Test rows: 5424
Training clients: 23
Test clients: 6
Client overlap: 0

FINAL WEEK-6 VALIDATION RESULT
Training clients: 23
Test clients: 6
Client overlap: 0
Honest grouped NDCG: 0.8157


,Method,NDCG
0,Week-4 baseline,0.439100
1,Week-5 Random Forest,0.462900
2,Week-6 Random Forest — honest grouped split,0.815744


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a **Random Forest Regressor** to estimate the next observed CTR from information available at the current observation.

This method fits the lane because the question is about identifying pages with potential future search-performance improvement. The warehouse does not contain a direct refresh-success label, so I will use the next observed CTR as a limited future-performance proxy rather than calling it a true refresh outcome.

Random Forest is appropriate because it can model nonlinear relationships between search visibility and engagement signals without requiring a linear relationship. I will keep the model simple and compare it directly with the transparent Week-4 baseline.

The model is used for **decision-support**, not causal inference. A higher predicted future CTR does not prove that refreshing a page will cause an improvement.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a **time-aware split** because the same content pages can appear on multiple report dates.

The target for each observation is the CTR on the next observed report date for the same client and content page. Therefore, the model must only use information available at the current report date.

I will use the earlier observed dates for training and the later observed dates for testing. This avoids using future observations to predict earlier observations.

The date gap from January 31 to February 10 is retained rather than treated as continuous daily history. The outcome means the next available observed date, not necessarily the following calendar day.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest will be trained only on observations from the earlier time period. The test set contains later report dates that were not used during training.

The comparison uses the same held-out observations. The model produces a predicted future CTR, while the Week-4 baseline produces its existing action score.

Because the baseline is a ranking score rather than a calibrated prediction, I will compare both methods primarily as ranking systems on the same test observations. I will also report the model's MAE and RMSE for the future-CTR prediction task.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's errors are examined using absolute prediction error rather than looking only at the overall metric.

Large errors indicate observations where the current measured signals did not accurately estimate the next observed CTR. These cases may reflect noisy low-impression observations, changes between observed dates, or other factors not represented in the available fields.

I will also use permutation importance to understand which observed features contribute most to the model's predictions. These are associations used for interpretation, not causal effects.


### Interpretation

The Week-5 Random Forest achieved an NDCG of **0.4629** on the held-out test period, compared with **0.4391** for the Week-4 baseline. This is an improvement of **0.0238 NDCG points**.

This indicates that, on this test set, the Random Forest produced a somewhat better ranking of observations by their next observed CTR than the transparent Week-4 baseline.

The improvement should be interpreted cautiously. The target is the **next observed CTR**, which is only a limited proxy for future search performance and is not a direct measure of refresh success. The model also does not establish that refreshing a page will cause CTR to increase.

The error analysis shows that prediction difficulty varies across observations. The mean absolute error was **0.0202**, while the RMSE was **0.0527**. The maximum absolute error was **0.9971**, showing that a small number of observations can have very large prediction errors. Low-impression observations may produce unstable CTR values because a small change in clicks can cause a large change in the observed rate.

Permutation importance identified **gsc_avg_position, gsc_clicks, ga4_engaged_sessions, ga4_sessions, and ga4_total_engagement_sec** as the five most important observed features. These are associations used to interpret the model and should not be treated as causal effects.

Overall, the Random Forest is useful as a **directional decision-support ranking model**. It performs better than the Week-4 baseline on this held-out test period, but the result should not be interpreted as evidence that the model or a content refresh itself causes future performance improvements.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.